# LeetCode 256 — Paint House bằng Python

Link đề: https://leetcode.com/problems/paint-house/ · Độ khó: **Medium** (Premium) · Chủ đề: Dynamic Programming, Array

Skeleton LeetCode:

```python
class Solution:
    def minCost(self, costs: List[List[int]]) -> int:
        ...
```

Ghi chú: đề bài viết theo hiểu biết về bài (bài Premium), nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (thử mọi cách tô màu `3^n` với `n` nhỏ).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một hàng `n` ngôi nhà, mỗi nhà tô **một trong ba màu** (đỏ, xanh dương, xanh lá) với chi phí mỗi màu mỗi nhà khác nhau; **hai nhà liền kề không cùng màu**; tìm **tổng chi phí nhỏ nhất**.
2. Nhận ra bài **quy hoạch động (DP)**: trạng thái = (nhà thứ `i`, màu của nhà `i`); chuyển trạng thái = nhà `i` tô màu `c` thì nhà `i-1` phải tô màu **khác `c`**.
3. Đi từ **vét cạn `3^n`** → **đệ quy có nhớ (top-down)** → **bảng DP `n × 3` (bottom-up)** → **3 biến cuộn `O(1)` bộ nhớ** → **tổng quát `k` màu bằng min/min thứ hai** (Paint House II).
4. Hiểu vì sao **tham lam (greedy)** chọn màu rẻ nhất cho từng nhà **sai**, và vì sao khi cập nhật nhiều biến DP cùng lúc phải **tính vế phải trước** (gán bộ).
5. Tránh các lỗi kinh điển: greedy, dùng giá trị **đã cập nhật** thay vì giá trị của nhà trước, quên lấy `min` ở cuối, cho phép hai nhà kề cùng màu (`min(a, b, c)`), khởi tạo sai.

## 2. Đề bài & ví dụ

Có một hàng `n` ngôi nhà, mỗi nhà có thể được sơn **một trong ba màu**: đỏ, xanh dương hoặc xanh lá. **Chi phí** sơn mỗi nhà bằng mỗi màu là khác nhau. Bạn phải sơn **tất cả** ngôi nhà sao cho **không có hai ngôi nhà liền kề cùng màu**.

Chi phí được cho bằng **ma trận `n × 3`** `costs`. Ví dụ `costs[0][0]` là chi phí sơn nhà `0` màu đỏ; `costs[1][2]` là chi phí sơn nhà `1` màu xanh lá, v.v.

Trả về **chi phí nhỏ nhất** để sơn tất cả các nhà.

**Ví dụ 1**

- Input: `costs = [[17,2,17],[16,16,5],[14,3,19]]`
- Output: `10`
- Giải thích: sơn nhà `0` màu xanh dương, nhà `1` màu xanh lá, nhà `2` màu xanh dương. Chi phí tối thiểu: `2 + 5 + 3 = 10`.

**Ví dụ 2**

- Input: `costs = [[7,6,2]]`
- Output: `2`

**Ràng buộc:** `costs.length == n`; `costs[i].length == 3`; `1 <= n <= 100`; `1 <= costs[i][j] <= 20`.

In [1]:
from typing import List
from functools import lru_cache
from itertools import product
import random
import time

examples = [
    ([[17, 2, 17], [16, 16, 5], [14, 3, 19]], 10),
    ([[7, 6, 2]], 2),
]
for costs, expected in examples:
    print(f"costs={costs} -> kỳ vọng {expected}")

costs=[[17, 2, 17], [16, 16, 5], [14, 3, 19]] -> kỳ vọng 10
costs=[[7, 6, 2]] -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Ma trận chi phí và chỉ số `costs[i][c]`

`costs[i]` là danh sách ba chi phí của nhà `i`; `costs[i][c]` với `c ∈ {0, 1, 2}` (đỏ, xanh dương, xanh lá) là chi phí tô nhà `i` màu `c`. `len(costs)` là `n`.
#### Code demo

In [2]:
costs = [[17, 2, 17], [16, 16, 5], [14, 3, 19]]
names = ["đỏ", "xanh dương", "xanh lá"]
for i, row in enumerate(costs):
    print(f"nhà {i}: " + ", ".join(f"{names[c]}={row[c]}" for c in range(3)))
print(len(costs), costs[1][2])      # 3 5

nhà 0: đỏ=17, xanh dương=2, xanh lá=17
nhà 1: đỏ=16, xanh dương=16, xanh lá=5
nhà 2: đỏ=14, xanh dương=3, xanh lá=19
3 5


### 3.2. Một cách tô màu hợp lệ và chi phí của nó

Một **cách tô** là dãy màu `colors[0..n-1]`. Nó **hợp lệ** nếu `colors[i] != colors[i+1]` với mọi `i`. Chi phí `= sum(costs[i][colors[i]])`. Ví dụ `colors = [1, 2, 1]` (xanh dương, xanh lá, xanh dương) có chi phí `2 + 5 + 3 = 10`.
#### Code demo

In [3]:
def cost_of(costs, colors):
    return sum(costs[i][c] for i, c in enumerate(colors))


def is_valid(colors):
    return all(colors[i] != colors[i + 1] for i in range(len(colors) - 1))


colors = [1, 2, 1]
print(is_valid(colors), cost_of(costs, colors))                 # True 10
print(is_valid([1, 1, 2]), cost_of(costs, [1, 1, 2]))           # False 37: có chi phí nhưng không hợp lệ nên không được dùng

True 10
False 37


### 3.3. Quy hoạch động: trạng thái và chuyển trạng thái

DP giải bài lớn bằng cách **lưu đáp án của bài con**. Ở đây gọi `dp[i][c]` là **chi phí nhỏ nhất để sơn các nhà `0..i`** biết rằng **nhà `i` sơn màu `c`**. Muốn tính `dp[i][c]`, nhà `i-1` phải sơn màu **khác `c`**, nên: `dp[i][c] = costs[i][c] + min(dp[i-1][c'] với c' ≠ c)`.
#### Code demo

In [4]:
dp0 = costs[0][:]                                              # nhà đầu: chưa có ràng buộc
print("dp[0] =", dp0)
dp1 = [costs[1][c] + min(dp0[c2] for c2 in range(3) if c2 != c) for c in range(3)]
print("dp[1] =", dp1)       # [16 + min(2, 17), 16 + min(17, 17), 5 + min(17, 2)] = [18, 33, 7]

dp[0] = [17, 2, 17]
dp[1] = [18, 33, 7]


### 3.4. Gán bộ (tuple assignment) và thứ tự tính vế phải

Câu lệnh `a, b, c = x, y, z` **tính xong toàn bộ vế phải** (`x`, `y`, `z` với các giá trị `a, b, c` **cũ**) rồi mới gán. Vì vậy `a, b, c = min(b, c) + ca, min(a, c) + cb, min(a, b) + cc` dùng đúng `a, b, c` cũ ở cả ba biểu thức. Nếu viết ba câu lệnh `a = ...`, `b = ...`, `c = ...` riêng thì `b` và `c` sẽ dùng `a` **đã bị đổi** — sai.
#### Code demo

In [5]:
a, b, c = 1, 2, 3
a, b, c = b, c, a                   # xoay vòng: vế phải dùng giá trị cũ
print(a, b, c)                      # 2 3 1

a, b, c = 1, 2, 3
a = b
b = a                               # b nhận a MỚI (= 2), không phải 1
print(a, b)                         # 2 2

2 3 1
2 2


### 3.5. `lru_cache` — nhớ kết quả hàm đệ quy

`@lru_cache(maxsize=None)` đặt trên hàm làm hàm **nhớ** kết quả theo đối số: gọi lại với cùng đối số trả ngay kết quả đã tính. Biến đệ quy "có nhiều bài con lặp" thành DP **top-down**. Đối số phải **băm được** (số, chuỗi, tuple).
#### Code demo

In [6]:
calls = 0


@lru_cache(maxsize=None)
def fib(n):
    global calls
    calls += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)


print(fib(30), "số lần thân hàm thật sự chạy:", calls)     # 832040, 31 (thay vì hơn một triệu lần)

832040 số lần thân hàm thật sự chạy: 31


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 1: `[[17,2,17],[16,16,5],[14,3,19]]`. Ta tính từ nhà đầu:

- Nhà `0`: chưa có ràng buộc, `dp[0] = [17, 2, 17]`.
- Nhà `1`:
  - đỏ (`16`): nhà `0` phải khác đỏ ⇒ `min(2, 17) = 2` ⇒ `18`;
  - xanh dương (`16`): nhà `0` khác xanh dương ⇒ `min(17, 17) = 17` ⇒ `33`;
  - xanh lá (`5`): nhà `0` khác xanh lá ⇒ `min(17, 2) = 2` ⇒ `7`.
  - `dp[1] = [18, 33, 7]`.
- Nhà `2`:
  - đỏ (`14`): `14 + min(33, 7) = 21`;
  - xanh dương (`3`): `3 + min(18, 7) = 10`;
  - xanh lá (`19`): `19 + min(18, 33) = 37`.
  - `dp[2] = [21, 10, 37]`.

Đáp án `min(dp[2]) = 10`. **Cách nghĩ:** để biết chi phí tốt nhất kết thúc ở một màu, chỉ cần biết chi phí tốt nhất của nhà trước ở **hai màu còn lại**.

## 5. Cách 1 — Brute Force / baseline: thử mọi cách tô màu

**Ý tưởng.** Sinh mọi dãy màu `colors ∈ {0,1,2}^n` (`3^n` dãy), loại các dãy có hai nhà kề cùng màu, tính chi phí, lấy nhỏ nhất. Dùng `itertools.product(range(3), repeat=n)`.
#### Code demo

In [7]:
def paint_bruteforce(costs):
    n = len(costs)
    best = float("inf")
    for colors in product(range(3), repeat=n):
        if all(colors[i] != colors[i + 1] for i in range(n - 1)):
            best = min(best, sum(costs[i][colors[i]] for i in range(n)))
    return best


for costs, expected in examples:
    print(costs, "->", paint_bruteforce(costs), "(kỳ vọng", expected, ")")

[[17, 2, 17], [16, 16, 5], [14, 3, 19]] -> 10 (kỳ vọng 10 )
[[7, 6, 2]] -> 2 (kỳ vọng 2 )


**Độ phức tạp (Cách 1).** Gọi `n` là số nhà: **Time `O(n · 3^n)`**; với `n = 100` là hàng `10^47` — **không thể**; chỉ chạy được tới `n ≈ 12`. **Space `O(n)`**.

**Nút thắt.** Rất nhiều dãy chia sẻ cùng một **tiền tố** và cùng màu ở nhà cuối; ta **tính lại** chi phí tối ưu của tiền tố đó nhiều lần. DP chỉ tính **một lần** cho mỗi (nhà, màu).

## 6. Ý tưởng thuật toán / bước đột phá

**Cấu trúc con tối ưu:** chi phí tối ưu để sơn các nhà `0..i` với nhà `i` màu `c` chỉ phụ thuộc vào **chi phí tối ưu của các nhà `0..i-1` theo từng màu của nhà `i-1`**, không phụ thuộc cách tô cụ thể trước đó. Đó chính là điều kiện để dùng DP:

`dp[i][c] = costs[i][c] + min(dp[i-1][c'] : c' ≠ c)`, `dp[0][c] = costs[0][c]`, đáp án `min(dp[n-1])`.

Vì `dp[i]` chỉ phụ thuộc `dp[i-1]`, ta **không cần bảng `n × 3`**: giữ **ba biến** `a, b, c` là `dp[i-1][0..2]` và cập nhật **đồng thời** (gán bộ). Thời gian `O(n)`, bộ nhớ `O(1)`.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ (top-down)

**Giải thích.** `f(i, c)` = chi phí nhỏ nhất để sơn nhà `i` **trở về trước** khi nhà `i` mang màu `c`. `f(0, c) = costs[0][c]`; `f(i, c) = costs[i][c] + min(f(i-1, c') với c' ≠ c)`. Đáp án `min(f(n-1, c))`.
#### Code demo

In [8]:
def paint_memo(costs):
    n = len(costs)

    @lru_cache(maxsize=None)
    def f(i, c):
        if i == 0:
            return costs[0][c]
        return costs[i][c] + min(f(i - 1, other) for other in range(3) if other != c)

    return min(f(n - 1, c) for c in range(3))


for costs, expected in examples:
    print(costs, "->", paint_memo(costs), "(kỳ vọng", expected, ")")

[[17, 2, 17], [16, 16, 5], [14, 3, 19]] -> 10 (kỳ vọng 10 )
[[7, 6, 2]] -> 2 (kỳ vọng 2 )


**Độ phức tạp Cách 2:** `3n` trạng thái, mỗi trạng thái `O(1)` (hai ứng viên): **Time `O(n)`**, **Space `O(n)`** (bộ nhớ đệm và ngăn xếp đệ quy sâu `n ≤ 100`).

### Cách 3 — Bảng DP `n × 3` (bottom-up)

**Giải thích.** Lập `dp` kích thước `n × 3`; điền từ trên xuống dưới theo công thức. Dễ đọc và dễ **truy vết** cách tô (nếu cần).
#### Code demo

In [9]:
def paint_table(costs):
    n = len(costs)
    dp = [[0] * 3 for _ in range(n)]
    dp[0] = costs[0][:]                                       # nhà đầu: chi phí đúng bằng chi phí tô
    for i in range(1, n):
        for c in range(3):
            dp[i][c] = costs[i][c] + min(dp[i - 1][other] for other in range(3) if other != c)
    return min(dp[-1])


for costs, expected in examples:
    print(costs, "->", paint_table(costs), "(kỳ vọng", expected, ")")

[[17, 2, 17], [16, 16, 5], [14, 3, 19]] -> 10 (kỳ vọng 10 )
[[7, 6, 2]] -> 2 (kỳ vọng 2 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** (`3n` ô).

### Cách 4 — Ba biến cuộn, `O(1)` bộ nhớ (cách nộp chuẩn)

**Giải thích.** `a, b, c` là `dp[i-1][đỏ], dp[i-1][xanh dương], dp[i-1][xanh lá]`. Với nhà mới có chi phí `(ca, cb, cc)`: `a' = ca + min(b, c)`, `b' = cb + min(a, c)`, `c' = cc + min(a, b)`. Gán **bộ** để cả ba dùng giá trị cũ.
#### Code demo

In [10]:
def paint_rolling(costs):
    a = b = c = 0                                 # trước nhà đầu tiên: chi phí 0 (không có nhà trước nên không có ràng buộc)
    for ca, cb, cc in costs:
        a, b, c = ca + min(b, c), cb + min(a, c), cc + min(a, b)
    return min(a, b, c)


for costs, expected in examples:
    print(costs, "->", paint_rolling(costs), "(kỳ vọng", expected, ")")

[[17, 2, 17], [16, 16, 5], [14, 3, 19]] -> 10 (kỳ vọng 10 )
[[7, 6, 2]] -> 2 (kỳ vọng 2 )


In [11]:
# Code nộp LeetCode cho Cách 4
class Solution:
    def minCost(self, costs: List[List[int]]) -> int:
        a = b = c = 0
        for ca, cb, cc in costs:
            a, b, c = min(b, c) + ca, min(a, c) + cb, min(a, b) + cc
        return min(a, b, c)


print(Solution().minCost([[17, 2, 17], [16, 16, 5], [14, 3, 19]]))   # 10

10


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`**. Khởi tạo `a = b = c = 0` là mẹo: với nhà đầu, `min(b, c) = 0` nên `a = ca` đúng như `dp[0]`.

### Cách 5 — Tổng quát `k` màu (Paint House II): min và min thứ hai

**Giải thích.** Với `k` màu, `min(dp[i-1][c'] : c' ≠ c)` có thể lấy `O(k)` mỗi màu (tổng `O(n·k²)`). Mẹo `O(n·k)`: lưu **giá trị nhỏ nhất** `m1` (và màu `idx1` đạt nó) cùng **giá trị nhỏ thứ hai** `m2`. Với màu `c`: nếu `c ≠ idx1` dùng `m1`, nếu `c == idx1` dùng `m2`.
#### Code demo

In [12]:
def paint_k_colors(costs):
    """Tổng quát k màu (k >= 2), O(n * k) nhờ min và min thứ hai của hàng trước."""
    n, k = len(costs), len(costs[0])
    prev = [0] * k
    for i in range(n):
        order = sorted(range(k), key=lambda c: prev[c])
        idx1, m1 = order[0], prev[order[0]]
        m2 = prev[order[1]]
        cur = [costs[i][c] + (m2 if c == idx1 else m1) for c in range(k)]
        prev = cur
    return min(prev)


for costs, expected in examples:
    if len(costs[0]) == 3:
        print(costs, "->", paint_k_colors(costs), "(kỳ vọng", expected, ")")

[[17, 2, 17], [16, 16, 5], [14, 3, 19]] -> 10 (kỳ vọng 10 )
[[7, 6, 2]] -> 2 (kỳ vọng 2 )


**Độ phức tạp Cách 5:** viết bằng `sorted` mỗi hàng: `O(n·k log k)` (có thể tối ưu `O(n·k)` bằng hai lần quét tìm min/min2); **Space `O(k)`**. Với `k = 3` thì tương đương Cách 4.

> Cách 4 là bản nộp chuẩn. Cách 3 hữu ích để truy vết. Cách 5 mở đường sang Paint House II (LeetCode 265).

## 8. TRACE từng bước

Ta in `dp` sau từng nhà (Cách 3) và cặp giá trị `a, b, c` (Cách 4) cho ví dụ 1; kèm lý do chọn `min`.
#### Code demo

In [13]:
def trace_dp(costs):
    n = len(costs)
    names = ["đỏ", "xanh dương", "xanh lá"]
    dp = [costs[0][:]]
    print(f"nhà 0: dp = {dp[0]}")
    for i in range(1, n):
        row = []
        for c in range(3):
            others = [(dp[i - 1][o], names[o]) for o in range(3) if o != c]
            best, from_name = min(others)
            row.append(costs[i][c] + best)
            print(f"  nhà {i}, {names[c]:>10}: {costs[i][c]} + min trên màu khác ({others}) = {costs[i][c]} + {best} (từ {from_name}) = {row[-1]}")
        dp.append(row)
        print(f"nhà {i}: dp = {row}")
    print("đáp án = min(dp[-1]) =", min(dp[-1]))
    return min(dp[-1])


trace_dp([[17, 2, 17], [16, 16, 5], [14, 3, 19]])

nhà 0: dp = [17, 2, 17]
  nhà 1,         đỏ: 16 + min trên màu khác ([(2, 'xanh dương'), (17, 'xanh lá')]) = 16 + 2 (từ xanh dương) = 18
  nhà 1, xanh dương: 16 + min trên màu khác ([(17, 'đỏ'), (17, 'xanh lá')]) = 16 + 17 (từ xanh lá) = 33
  nhà 1,    xanh lá: 5 + min trên màu khác ([(17, 'đỏ'), (2, 'xanh dương')]) = 5 + 2 (từ xanh dương) = 7
nhà 1: dp = [18, 33, 7]
  nhà 2,         đỏ: 14 + min trên màu khác ([(33, 'xanh dương'), (7, 'xanh lá')]) = 14 + 7 (từ xanh lá) = 21
  nhà 2, xanh dương: 3 + min trên màu khác ([(18, 'đỏ'), (7, 'xanh lá')]) = 3 + 7 (từ xanh lá) = 10
  nhà 2,    xanh lá: 19 + min trên màu khác ([(18, 'đỏ'), (33, 'xanh dương')]) = 19 + 18 (từ đỏ) = 37
nhà 2: dp = [21, 10, 37]
đáp án = min(dp[-1]) = 10


10

In [14]:
def trace_rolling(costs):
    a = b = c = 0
    print(f"bắt đầu: a={a}, b={b}, c={c}")
    for i, (ca, cb, cc) in enumerate(costs):
        na, nb, nc = ca + min(b, c), cb + min(a, c), cc + min(a, b)
        print(f"nhà {i}: ({ca},{cb},{cc}): a'={ca}+min({b},{c}) = {na}, b'={cb}+min({a},{c}) = {nb}, c'={cc}+min({a},{b}) = {nc}")
        a, b, c = na, nb, nc
    print("đáp án =", min(a, b, c))
    return min(a, b, c)


trace_rolling([[17, 2, 17], [16, 16, 5], [14, 3, 19]])

bắt đầu: a=0, b=0, c=0
nhà 0: (17,2,17): a'=17+min(0,0) = 17, b'=2+min(0,0) = 2, c'=17+min(0,0) = 17
nhà 1: (16,16,5): a'=16+min(2,17) = 18, b'=16+min(17,17) = 33, c'=5+min(17,2) = 7
nhà 2: (14,3,19): a'=14+min(33,7) = 21, b'=3+min(18,7) = 10, c'=19+min(18,33) = 37
đáp án = 10


10

Bảng tóm tắt (ví dụ 1):

| nhà | chi phí `(đỏ, xanh dương, xanh lá)` | `dp` sau nhà đó |
|---|---|---|
| 0 | `(17, 2, 17)` | `[17, 2, 17]` |
| 1 | `(16, 16, 5)` | `[18, 33, 7]` |
| 2 | `(14, 3, 19)` | `[21, 10, 37]` |

Đáp án `min(21, 10, 37) = 10`: cách tô xanh dương (2), xanh lá (5), xanh dương (3).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (ba biến cuộn). Đúng `class Solution`, đúng `minCost(self, costs: List[List[int]]) -> int`.

In [15]:
class Solution:
    def minCost(self, costs: List[List[int]]) -> int:
        a = b = c = 0                                      # chi phí tối ưu kết thúc ở đỏ / xanh dương / xanh lá (trước nhà đầu: 0)
        for ca, cb, cc in costs:
            # gán bộ: cả ba biểu thức dùng a, b, c CŨ (của nhà trước)
            a, b, c = min(b, c) + ca, min(a, c) + cb, min(a, b) + cc
        return min(a, b, c)


sol = Solution()
for costs, expected in examples:
    assert sol.minCost(costs) == expected
print("OK: 2 ví dụ của đề")

OK: 2 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Cấu trúc con tối ưu:** gọi `OPT(i, c)` là chi phí nhỏ nhất để sơn hợp lệ nhà `0..i` với nhà `i` màu `c`. Một cách tô tối ưu cho `(i, c)` có nhà `i-1` màu `c' ≠ c` nào đó; phần sơn nhà `0..i-1` **phải tối ưu cho `(i-1, c')`** (nếu không, thay bằng cách tối ưu thì tổng nhỏ hơn mà vẫn hợp lệ vì ràng buộc giữa nhà `i-1` và `i` chỉ phụ thuộc màu `c'`). Do đó `OPT(i, c) = costs[i][c] + min_{c' ≠ c} OPT(i-1, c')`.
- **Cơ sở:** `OPT(0, c) = costs[0][c]`. Khởi tạo `a = b = c = 0` ở Cách 4 cho cùng kết quả vì `min(0, 0) = 0`.
- **Đáp án:** nhà cuối có thể màu nào cũng được ⇒ `min` trên ba màu.
- **Gán bộ:** bảo đảm `OPT(i, ·)` chỉ dùng `OPT(i-1, ·)`.
- **Vì sao greedy sai:** chọn màu rẻ nhất cho nhà hiện tại có thể **chặn** một màu rẻ ở nhà kế tiếp (xem phản ví dụ ở mục 12.2).

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nhà (`<= 100`), `k = 3` là số màu.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n · 3^n)` | `O(n)` |
| 2. Đệ quy có nhớ | `O(n·k²)` | `O(n·k)` |
| 3. Bảng DP | `O(n·k²)` | `O(n·k)` |
| 4. Ba biến cuộn | `O(n)` (`k = 3`) | `O(1)` |
| 5. `k` màu, min/min thứ hai | `O(n·k log k)` (`O(n·k)` nếu tối ưu) | `O(k)` |

Với `n = 100`, Cách 2–4 chỉ vài trăm phép — tức thời; Cách 1 không chạy nổi. Không có khác biệt average/worst (mọi nhánh luôn duyệt đủ `n` nhà).

## 12. Bộ test

`oracle` là brute force độc lập: **thử mọi dãy màu** bằng đệ quy chọn từng nhà (không dùng `itertools.product`, không DP), dùng được cho `n <= 9`. Với `n` lớn, đối chiếu các cách DP với nhau và với một **DP bằng đồ thị** (đường đi ngắn nhất trên đồ thị phân tầng) độc lập.
#### Code demo

In [16]:
def oracle(costs):
    """Brute force độc lập: chọn màu từng nhà bằng đệ quy, ràng buộc khác màu nhà trước (chỉ cho n nhỏ)."""
    n = len(costs)
    best = float("inf")

    def go(i, prev_color, total):
        nonlocal best
        if total >= best:
            return                                    # cắt nhánh (không ảnh hưởng tính đúng vì chi phí dương)
        if i == n:
            best = total
            return
        for c in range(3):
            if c != prev_color:
                go(i + 1, c, total + costs[i][c])

    go(0, -1, 0)
    return best


impls = {
    "bruteforce": paint_bruteforce,
    "memo": paint_memo,
    "table": paint_table,
    "rolling": paint_rolling,
    "k_colors": paint_k_colors,
    "submit": sol.minCost,
}

for costs, expected in examples:
    assert oracle(costs) == expected
    for key, fn in impls.items():
        assert fn(costs) == expected, (key, costs, fn(costs), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [17]:
edge_cases = [
    ([[1, 1, 1]], 1, "một nhà: màu rẻ nhất"),
    ([[20, 20, 20]], 20, "một nhà: giá biên"),
    ([[1, 2, 3], [1, 2, 3]], 3, "hai nhà: không thể cùng màu rẻ nhất (1 + 2)"),
    ([[5, 5, 5], [5, 5, 5], [5, 5, 5]], 15, "mọi chi phí bằng nhau"),
    ([[1, 20, 20], [1, 20, 20], [1, 20, 20]], 1 + 20 + 1, "chỉ một màu rẻ: phải xen kẽ"),
    ([[1, 2, 3], [2, 100, 100]], 4, "greedy sai: nhà 0 màu 1 (chi phí 2), nhà 1 màu 0 (chi phí 2)"),
    ([[20, 1, 20], [1, 20, 20], [20, 1, 20], [1, 20, 20]], 4, "xen kẽ hai màu rẻ"),
    ([[1, 1, 1]] * 100, 100, "n = 100, mọi chi phí 1"),
    ([[20, 20, 20]] * 100, 2000, "n = 100, mọi chi phí 20: cận trên của đề"),
]
for costs, expected, label in edge_cases:
    if len(costs) <= 9:
        assert oracle(costs) == expected, (label, oracle(costs), expected)
    for key, fn in impls.items():
        if key == "bruteforce" and len(costs) > 9:
            continue
        assert fn(costs) == expected, (key, label, fn(costs), expected)
    print("PASS", label)

PASS một nhà: màu rẻ nhất
PASS một nhà: giá biên
PASS hai nhà: không thể cùng màu rẻ nhất (1 + 2)
PASS mọi chi phí bằng nhau
PASS chỉ một màu rẻ: phải xen kẽ
PASS greedy sai: nhà 0 màu 1 (chi phí 2), nhà 1 màu 0 (chi phí 2)
PASS xen kẽ hai màu rẻ
PASS n = 100, mọi chi phí 1
PASS n = 100, mọi chi phí 20: cận trên của đề


### 12.2. Tính chất và phản ví dụ

- Đáp án `>= n` (mỗi nhà tốn ít nhất `1`) và `<= 20n`.
- Đáp án `>= tổng của min mỗi nhà` (cận dưới "bỏ ràng buộc"); và nếu **một màu luôn rẻ nhất ở mọi nhà** thì cần xen kẽ nên đáp án thường lớn hơn cận dưới.
- Cộng cùng một hằng `d` vào **mọi** chi phí thì đáp án tăng đúng `n·d`.
- Hoán đổi cột (đổi tên màu) không đổi đáp án; **đảo thứ tự nhà** không đổi đáp án.
- **Phản ví dụ** cho greedy: `[[1,2,3],[2,100,100]]` có đáp án `4` nhưng greedy trả `101`.
#### Code demo

In [18]:
random.seed(256)
for _ in range(500):
    n = random.randint(1, 9)
    costs = [[random.randint(1, 20) for _ in range(3)] for _ in range(n)]
    ans = sol.minCost(costs)
    assert n <= ans <= 20 * n
    assert ans >= sum(min(r) for r in costs)
    d = random.randint(1, 5)
    assert sol.minCost([[x + d for x in r] for r in costs]) == ans + n * d
    perm = random.sample(range(3), 3)
    assert sol.minCost([[r[p] for p in perm] for r in costs]) == ans
    assert sol.minCost(costs[::-1]) == ans
print("PASS tính chất (cận, cộng hằng, hoán đổi màu, đảo thứ tự nhà) trên 500 bộ ngẫu nhiên")


def sai_tham_lam(costs):
    total, prev = 0, -1
    for row in costs:
        c = min((x, i) for i, x in enumerate(row) if i != prev)[1]       # SAI: chọn màu rẻ nhất cho nhà hiện tại
        total += row[c]
        prev = c
    return total


bad = [[1, 2, 3], [2, 100, 100]]
print("phản ví dụ:", bad, "-> đúng", sol.minCost(bad), "| tham lam:", sai_tham_lam(bad))
assert sai_tham_lam(bad) != sol.minCost(bad)

PASS tính chất (cận, cộng hằng, hoán đổi màu, đảo thứ tự nhà) trên 500 bộ ngẫu nhiên
phản ví dụ: [[1, 2, 3], [2, 100, 100]] -> đúng 4 | tham lam: 101


### 12.3. Đối chiếu ngẫu nhiên (có seed) và `n` lớn
#### Code demo

In [19]:
random.seed(4256)
for trial in range(800):
    n = random.randint(1, 9)
    hi = random.choice([2, 5, 20])
    costs = [[random.randint(1, hi) for _ in range(3)] for _ in range(n)]
    truth = oracle(costs)
    for key, fn in impls.items():
        assert fn(costs) == truth, (key, costs, fn(costs), truth)
print("PASS 800 bộ ngẫu nhiên nhỏ (n <= 9): mọi cách khớp oracle")


def shortest_path_layered(costs):
    """DP độc lập dạng đường đi ngắn nhất trên đồ thị phân tầng: đỉnh (i, c), cạnh (i-1, c') -> (i, c) nếu c' != c, trọng số costs[i][c]."""
    n = len(costs)
    INF = float("inf")
    dist = {(0, c): costs[0][c] for c in range(3)}
    for i in range(1, n):
        for c in range(3):
            dist[(i, c)] = min(dist[(i - 1, c2)] + costs[i][c] for c2 in range(3) if c2 != c)
    return min(dist[(n - 1, c)] for c in range(3))


for trial in range(300):
    n = random.randint(10, 100)
    costs = [[random.randint(1, 20) for _ in range(3)] for _ in range(n)]
    ref = shortest_path_layered(costs)
    for key in ("memo", "table", "rolling", "k_colors", "submit"):
        assert impls[key](costs) == ref, (key, n)
print("PASS 300 bộ n tới 100: các cách DP khớp đồ thị phân tầng")

PASS 800 bộ ngẫu nhiên nhỏ (n <= 9): mọi cách khớp oracle
PASS 300 bộ n tới 100: các cách DP khớp đồ thị phân tầng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tham lam (greedy): chọn màu rẻ nhất cho từng nhà
Xem `sai_tham_lam` ở phần test: `[[1,2,3],[2,100,100]]` trả `101` thay vì `4`. **Vì sao sai:** chọn rẻ nhất ở nhà `0` (màu `0`, chi phí `1`) chặn màu rẻ `2` của nhà `1`, buộc nhà `1` chọn `100`. **Code đúng:** DP xét mọi màu của nhà trước.

### Lỗi 2: Cập nhật tuần tự (dùng giá trị mới của `a`)
#### Code sai

In [20]:
def sai_cap_nhat_tuan_tu(costs):
    a = b = c = 0
    for ca, cb, cc in costs:
        a = min(b, c) + ca
        b = min(a, c) + cb                         # SAI: a đã là giá trị MỚI của nhà hiện tại
        c = min(a, b) + cc                         # SAI: a, b đều là giá trị mới
    return min(a, b, c)


for costs in ([[17, 2, 17], [16, 16, 5], [14, 3, 19]], [[1, 2, 3], [2, 100, 100]]):
    print(costs, "đúng:", sol.minCost(costs), "| sai:", sai_cap_nhat_tuan_tu(costs))

[[17, 2, 17], [16, 16, 5], [14, 3, 19]] đúng: 10 | sai: 26
[[1, 2, 3], [2, 100, 100]] đúng: 4 | sai: 4


**Vì sao sai:** `dp[i][·]` phải dựa **hoàn toàn** vào `dp[i-1][·]`. Dùng `a` mới cho `b` làm hai nhà liền kề (cùng nhà `i`) "ràng buộc lẫn nhau" vô lý. **Code đúng:** gán bộ `a, b, c = ..., ..., ...`.

### Lỗi 3: Cho phép hai nhà kề cùng màu (`min(a, b, c)`)
#### Code sai

In [21]:
def sai_cho_phep_cung_mau(costs):
    a = b = c = 0
    for ca, cb, cc in costs:
        m = min(a, b, c)                           # SAI: không loại màu của nhà trước
        a, b, c = m + ca, m + cb, m + cc
    return min(a, b, c)


for costs in ([[17, 2, 17], [16, 16, 5], [14, 3, 19]], [[1, 2, 3], [1, 100, 100]]):
    print(costs, "đúng:", sol.minCost(costs), "| sai:", sai_cho_phep_cung_mau(costs))

[[17, 2, 17], [16, 16, 5], [14, 3, 19]] đúng: 10 | sai: 10
[[1, 2, 3], [1, 100, 100]] đúng: 3 | sai: 2


**Vì sao sai:** bản này trả `tổng các min của từng nhà` — chính là cận dưới "bỏ ràng buộc". **Code đúng:** với màu `c` chỉ lấy `min` trên **hai màu khác**.

### Lỗi 4: Quên lấy `min` ở cuối (hoặc lấy `min` của cả bảng)
#### Code sai

In [22]:
def sai_quen_min_cuoi(costs):
    a = b = c = 0
    for ca, cb, cc in costs:
        a, b, c = min(b, c) + ca, min(a, c) + cb, min(a, b) + cc
    return a                                       # SAI: chỉ xét nhà cuối tô màu đỏ


for costs in ([[17, 2, 17], [16, 16, 5], [14, 3, 19]], [[7, 6, 2]]):
    print(costs, "đúng:", sol.minCost(costs), "| sai:", sai_quen_min_cuoi(costs))

[[17, 2, 17], [16, 16, 5], [14, 3, 19]] đúng: 10 | sai: 21
[[7, 6, 2]] đúng: 2 | sai: 7


**Vì sao sai:** nhà cuối có thể sơn **bất kỳ** màu nào; đáp án là `min(a, b, c)`.

### Lỗi 5: Khởi tạo sai (bắt đầu từ `dp[0]` nhưng lặp từ `0`)
#### Code sai

In [23]:
def sai_khoi_tao(costs):
    a, b, c = costs[0]
    for ca, cb, cc in costs:                       # SAI: lặp lại cả nhà đầu tiên (đếm nhà 0 hai lần)
        a, b, c = min(b, c) + ca, min(a, c) + cb, min(a, b) + cc
    return min(a, b, c)


for costs in ([[7, 6, 2]], [[17, 2, 17], [16, 16, 5], [14, 3, 19]]):
    print(costs, "đúng:", sol.minCost(costs), "| sai:", sai_khoi_tao(costs))

[[7, 6, 2]] đúng: 2 | sai: 8
[[17, 2, 17], [16, 16, 5], [14, 3, 19]] đúng: 10 | sai: 27


**Vì sao sai:** khởi tạo bằng chi phí nhà `0` rồi vòng lặp lại duyệt nhà `0` lần nữa nên nhà `0` bị tô hai lần (thêm một "nhà ảo"). **Code đúng:** hoặc khởi tạo `a = b = c = 0` và duyệt mọi nhà, hoặc khởi tạo `dp = costs[0]` rồi duyệt `costs[1:]`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**tô màu / chọn trạng thái cho dãy, ràng buộc giữa hai phần tử liền kề, tối thiểu hóa tổng chi phí**" → **DP theo vị trí và trạng thái cuối**.
- Công thức: `dp[i][c] = cost[i][c] + min(dp[i-1][c'] , c' ≠ c)`.
- `dp[i]` chỉ phụ thuộc `dp[i-1]` ⇒ **mảng cuộn / vài biến**.
- Cập nhật nhiều biến cùng lúc → **gán bộ** (tính vế phải trước).
- **Greedy** hiếm khi đúng với ràng buộc kề nhau: luôn thử phản ví dụ nhỏ.
- Tổng quát `k` màu → giữ **min** và **min thứ hai** của hàng trước để cập nhật `O(k)`.
- Kiểm tra: một nhà (`min` hàng), hai nhà (tránh cùng màu rẻ nhất), mọi chi phí bằng nhau (đáp án `n·giá`).

## 15. Biến thể của bài

1. **Paint House II** (LeetCode 265): `k` màu — Cách 5.
2. **Paint House III** (1473): một số nhà đã sơn sẵn, cần đúng `target` nhóm — DP thêm chiều.
3. **Paint Fence** (276): đếm số cách sơn hàng rào không có ba cột liên tiếp cùng màu.
4. **House Robber** (198), **Delete and Earn** (740): DP kề nhau tương tự.
5. **Truy vết** cách tô tối ưu (trả cả dãy màu).
6. **Đếm** số cách tô hợp lệ (không tối ưu): `3 · 2^(n-1)`.
#### Code demo

In [24]:
def paint_with_path(costs):
    """Bảng DP kèm con trỏ ngược để truy vết dãy màu tối ưu."""
    n = len(costs)
    dp = [costs[0][:]]
    parent = [[-1] * 3]
    for i in range(1, n):
        row, par = [], []
        for c in range(3):
            best_prev = min((dp[i - 1][o], o) for o in range(3) if o != c)
            row.append(costs[i][c] + best_prev[0])
            par.append(best_prev[1])
        dp.append(row)
        parent.append(par)
    c = min(range(3), key=lambda x: dp[-1][x])
    total = dp[-1][c]
    colors = [c]
    for i in range(n - 1, 0, -1):
        c = parent[i][c]
        colors.append(c)
    return total, colors[::-1]


total, colors = paint_with_path([[17, 2, 17], [16, 16, 5], [14, 3, 19]])
print(total, colors)                              # 10 [1, 2, 1]
assert cost_of([[17, 2, 17], [16, 16, 5], [14, 3, 19]], colors) == total and is_valid(colors)
print([3 * 2 ** (n - 1) for n in range(1, 6)])    # số cách tô hợp lệ: [3, 6, 12, 24, 48]

10 [1, 2, 1]
[3, 6, 12, 24, 48]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi `3^n` cách | `O(n·3^n)` | `O(n)` | oracle cho `n <= 9` |
| 2. Đệ quy có nhớ | `f(i, c)` + `lru_cache` | `O(n)` | `O(n)` | viết nhanh từ công thức |
| 3. Bảng DP | `dp[n][3]` từ trên xuống | `O(n)` | `O(n)` | cần truy vết cách tô |
| 4. Ba biến cuộn | `a, b, c` + gán bộ | `O(n)` | `O(1)` | **cách nộp chuẩn** |
| 5. `k` màu | min và min thứ hai | `O(n·k)` | `O(k)` | Paint House II |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `Solution().minCost([[3, 5, 7]])`; (b) `a, b = 1, 2; a, b = b, a + b` cho `a` và `b`; (c) `Solution().minCost([[1, 2, 3], [1, 2, 3]])`; (d) số cách tô hợp lệ cho `n = 3` (không xét chi phí).

**Bài 2 (tính bằng tay).** Tính bảng `dp` cho `costs = [[5, 8, 6], [19, 14, 13], [7, 5, 12]]` và cách tô tối ưu.

**Bài 3 (điền TODO).** Hoàn thiện `paint_todo(costs)` theo Cách 4.
#### Code demo

In [25]:
def paint_todo(costs):
    a = b = c = 0
    # TODO 1: duyệt (ca, cb, cc) trong costs
    # TODO 2: gán bộ: a, b, c = min(b, c) + ca, min(a, c) + cb, min(a, b) + cc
    # TODO 3: trả min(a, b, c)
    pass


# Tự làm xong rồi mới mở test:
# for costs, expected in examples:
#     assert paint_todo(costs) == expected

**Bài 4 (tự viết lại).** Viết `paint_memo2(costs)` bằng `lru_cache` **không nhìn lại** Cách 2; kiểm tra bằng `oracle` trên `n <= 8`.

**Bài 5 (biến thể / mini challenge).** Viết `paint_house_ii(costs)` cho `k` màu bằng hai lần quét tìm `min1`, `min2` mỗi hàng (thời gian `O(n·k)`), rồi viết `paint_fence(n, k)` đếm số cách sơn `n` cột bằng `k` màu sao cho **không có ba cột liên tiếp cùng màu** (gợi ý: `same[i]`, `diff[i]`). Thử `paint_fence(3, 2) == 6`, `paint_fence(7, 2) == 42`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: một nhà thì chọn màu rẻ nhất; gán bộ tính vế phải trước. Bài 2: dùng công thức `dp[i][c] = cost + min(hai màu khác)`. Bài 5: `paint_fence`: `same` = số cách mà hai cột cuối cùng màu, `diff` = khác màu.
- **Mức 2 (vừa):** Bài 1: `3`, `(2, 3)`, `3` (`1 + 2`), `12`. Bài 2: `dp[0] = [5, 8, 6]`; `dp[1] = [19 + min(8,6) = 25, 14 + min(5,6) = 19, 13 + min(5,8) = 18]`; `dp[2] = [7 + min(19,18) = 25, 5 + min(25,18) = 23, 12 + min(25,19) = 31]`; đáp án `23`. Bài 3: khung Cách 4.
- **Mức 3 (sâu):** Bài 2: truy vết `23`: nhà `2` xanh dương (`5`) từ nhà `1` xanh lá (`13`), từ nhà `0` đỏ (`5`) ⇒ `5 + 13 + 5 = 23`. Bài 5: `paint_fence`: `same' = diff`, `diff' = (same + diff) * (k - 1)`; khởi tạo `same = 0`, `diff = k` cho `n = 1`; đáp án `same + diff`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [26]:
# Bài 1
assert Solution().minCost([[3, 5, 7]]) == 3
a, b = 1, 2
a, b = b, a + b
assert (a, b) == (2, 3)
assert Solution().minCost([[1, 2, 3], [1, 2, 3]]) == 3
assert sum(1 for colors in product(range(3), repeat=3) if is_valid(list(colors))) == 12

# Bài 2
c2 = [[5, 8, 6], [19, 14, 13], [7, 5, 12]]
assert Solution().minCost(c2) == 23 == oracle(c2)
assert paint_with_path(c2) == (23, [0, 2, 1])


# Bài 3
def paint_todo_ans(costs):
    a = b = c = 0
    for ca, cb, cc in costs:
        a, b, c = min(b, c) + ca, min(a, c) + cb, min(a, b) + cc
    return min(a, b, c)


for costs, expected in examples:
    assert paint_todo_ans(costs) == expected


# Bài 4
def paint_memo2(costs):
    n = len(costs)

    @lru_cache(maxsize=None)
    def f(i, c):
        if i == 0:
            return costs[0][c]
        return costs[i][c] + min(f(i - 1, o) for o in range(3) if o != c)

    return min(f(n - 1, c) for c in range(3))


random.seed(11)
for _ in range(200):
    n = random.randint(1, 8)
    cs = [[random.randint(1, 20) for _ in range(3)] for _ in range(n)]
    assert paint_memo2(cs) == oracle(cs)


# Bài 5
def paint_house_ii(costs):
    k = len(costs[0])
    prev = [0] * k
    for row in costs:
        min1 = min2 = float("inf")
        idx1 = -1
        for c, v in enumerate(prev):
            if v < min1:
                min2, min1, idx1 = min1, v, c
            elif v < min2:
                min2 = v
        prev = [row[c] + (min2 if c == idx1 else min1) for c in range(k)]
    return min(prev)


for _ in range(200):
    n = random.randint(1, 8)
    cs = [[random.randint(1, 20) for _ in range(3)] for _ in range(n)]
    assert paint_house_ii(cs) == oracle(cs)


def paint_fence(n, k):
    if n == 1:
        return k
    same, diff = 0, k
    for _ in range(n - 1):
        same, diff = diff, (same + diff) * (k - 1)
    return same + diff


assert paint_fence(3, 2) == 6 and paint_fence(7, 2) == 42
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (ba biến cuộn + gán bộ)

```python
class Solution:
    def minCost(self, costs: List[List[int]]) -> int:
        a = b = c = 0
        for ca, cb, cc in costs:
            a, b, c = min(b, c) + ca, min(a, c) + cb, min(a, b) + cc
        return min(a, b, c)
```

**Checklist**

- [ ] `dp[i][c] = cost[i][c] + min(dp[i-1][c'] với c' ≠ c)` — chỉ lấy `min` trên **hai màu khác**?
- [ ] Dùng **gán bộ** để ba biểu thức đều dựa vào giá trị **cũ**?
- [ ] Khởi tạo `a = b = c = 0` và duyệt **mọi** nhà (hoặc khởi tạo `costs[0]` và duyệt `costs[1:]`)?
- [ ] Trả `min(a, b, c)` ở cuối?
- [ ] Không dùng greedy; đã thử phản ví dụ `[[1,2,3],[2,100,100]]`?
- [ ] Đã đối chiếu với vét cạn `3^n` cho `n <= 9`?
- [ ] Skeleton đúng `class Solution` / `def minCost(self, costs: List[List[int]]) -> int`?